In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import gc
from pathlib import Path
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import FuncFormatter
from IPython.display import display

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from foodcast.tools.coverage_functions import plot_time_series
from foodcast.tools.labeling_functions import plot_dish_time_series

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'(_sales_and_menu)?\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
before_after_details_true = pd.read_csv('data/3_data_parquet_relabeled/before_after_details_true.csv', index_col='location_id').assign(cross_over_date = lambda df: pd.to_datetime(df['cross_over_date']))

# Timezones
timezones = pd.read_csv('data/3_data_parquet_relabeled/timezones.csv', index_col='location_id')['timezone'].to_dict()
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])
    sales_and_menu_data[loc_id] = df
 
# Ordering (just good ones)
restaurants_by_4m_coverage = pd.read_csv('data/3_data_parquet_relabeled/restaurants_by_4m_coverage.csv')['location_id'].tolist()

# locations = list(sales_and_menu_data.keys())
# for other_loc_id in locations:
#     if other_loc_id != loc_id:
#         del sales_and_menu_data[other_loc_id]
#         del sales_data_merged[other_loc_id]
# gc.collect()

bad_restaurants = ['AQD04SM0J92WA', 'LBMCPAYT7W36V', 'L3XS7WSJ4AJA3', '1G5AJ17XCH2A8', '3AXDVZJYN9DRS', 'MS8R16DY0JQAM', 'N0PC58FB2XAZ3', 'ADPFRN3QZRCXK', 'WJA3YCD4QBWRX', '0RJH3FFPYBPEY', 'LZ5MR1TS37E7W']
# good_restaurants_by_4m_coverage = restaurants_by_4m_coverage.copy()
# for loc_id in bad_restaurants:
#     del sales_and_menu_data[loc_id]
#     del sales_data_merged[loc_id]
#     good_restaurants_by_4m_coverage.remove(loc_id)
# gc.collect()

# for loc in bad_restaurants:
#     restaurants_by_4m_coverage.remove(loc)

In [ ]:
for loc_id in restaurants_by_4m_coverage:
    print(loc_id, sales_and_menu_data[loc_id].index[0].year, sales_and_menu_data[loc_id].index[-1].year)

In [ ]:
def get_potential_confounders(sales_and_menu_data, before_after_details_true, organization = restaurants_by_4m_coverage, days=7, k=10):

    potential_confounders = {}
    potential_confounders_data = {}
    avg_min_sales = 0
    before_after_details_true = before_after_details_true.loc[organization]
    for loc_id, row in before_after_details_true.iterrows():
        
        df = sales_and_menu_data[loc_id]
        
        dt = pd.DateOffset(days=days)
        promo_date = row['cross_over_date']
        t1 = promo_date - dt
        t2 = promo_date + dt
        
        item_sales = df.groupby('item_name')['item_quantity'].sum().sort_values(ascending=False)
        total_items = item_sales.size
        total_sales = item_sales.sum()
        minimum_sales = round(k / total_sales * 1e5)
        avg_min_sales += minimum_sales
        
        # print(minimum_sales)
        
        item_first_dates = df.reset_index().groupby('item_name')['created_at'].first()
        items_introduced_then = item_first_dates.to_frame().query('@t1 < created_at < @t2').index.tolist()
        
        intervention_data = df.loc[t1:t2]
        intervention_data_introduced_then = intervention_data.query('item_name in @items_introduced_then')
        item_sales_introduced_then = intervention_data_introduced_then.groupby('item_name')['item_quantity'].sum().sort_values(ascending=False)
        
        nonneglible_item_sales = (item_sales_introduced_then
                                  .to_frame()
                                  #.pipe(lambda df: print(df.shape[0]) or df)
                                  .query('@k < item_quantity')
                                  #.pipe(lambda df: print(str(df.shape[0]) + "\n") or df)
                                  )
        potential_confounder_list = nonneglible_item_sales.index.tolist()
        
        nonneglible_item_sales_adjusted = (item_sales_introduced_then
                                           .to_frame()
                                           #.pipe(lambda df: print(df.shape[0]) or df)
                                           .query('@minimum_sales < item_quantity')
                                           #.pipe(lambda df: print(str(df.shape[0]) + "\n") or df)
                                           )
        potential_confounder_list_adjusted = nonneglible_item_sales_adjusted.index.tolist()
        
        potential_confounders[loc_id] = potential_confounder_list, potential_confounder_list_adjusted, total_items
        potential_confounders_data[loc_id] = df.query('item_name in @potential_confounder_list_adjusted') # using method 2
        
    s1 = pd.Series({k: len(v[0]) for k, v in potential_confounders.items()}, name='unadjusted, k>10')
    s2 = pd.Series({k: len(v[1]) for k, v in potential_confounders.items()}, name='adjusted by sales')
    s3 = pd.Series({k: round(len(v[0])/v[2] * 300) for k, v in potential_confounders.items()}, name='adjusted by menu')
    s4 = pd.Series({k: v[1] for k, v in potential_confounders.items()}, name='items')
    potential_confounder_coverage_details = pd.concat([s1,s2,s3,s4], axis=1).loc[organization]
    
    # print(avg_min_sales)
    
    return potential_confounder_coverage_details, potential_confounders_data

In [ ]:
# pd.set_option('display.max_colwidth', None) # default is 50

# days_for_visual = 28
# list_of_days = [7, 14, 28]
# data_to_plot = None
# for days in list_of_days:
#     output, potential_confounders_data = get_potential_confounders(sales_and_menu_data, days=days, k=10)
#     if days == days_for_visual:
#         data_to_plot = potential_confounders_data
#     display(output)

# for loc_id in good_restaurants_by_4m_coverage:
#     data = data_to_plot[loc_id]
#     plot_dish_time_series(data, loc_id, before_after_details_true)

In [ ]:
# Add rows
d1 = pd.DataFrame([{'location_id': 'ED5J990H5VAZT',
                                    'cross_over_date': pd.Timestamp('2019-12-14').tz_localize('UTC'),
                                    'batch':np.nan, 
                                    'first_plant_based_mention': 'Vegan',
                                    'promo_name': 'Vegan'}]).set_index('location_id')
d2 = pd.DataFrame([{'location_id': 'ED5J990H5VAZT',
                                    'cross_over_date': pd.Timestamp('2020-01-19').tz_localize('UTC'),
                                    'batch':np.nan, 
                                    'first_plant_based_mention': 'Vegan Egg',
                                    'promo_name': 'Vegan Egg'}]).set_index('location_id')
d3 = pd.DataFrame([{'location_id': 'ED5J990H5VAZT',
                                    'cross_over_date': pd.Timestamp('2020-04-26').tz_localize('UTC'), 
                                    'batch':np.nan, 
                                    'first_plant_based_mention': 'Vegan Sausage', 
                                    'promo_name': 'Vegan Sausage'}]).set_index('location_id')
d4 = pd.DataFrame([{'location_id': 'JHDN7CF1C03X5',
                                    'cross_over_date': pd.Timestamp('2020-03-12').tz_localize('UTC'), 
                                    'batch':np.nan, 
                                    'first_plant_based_mention': 'Beyond Sausage', 
                                    'promo_name': 'Beyond Sausage'}]).set_index('location_id')      
d5 = pd.DataFrame([{'location_id': 'W8T41JZK0ZMEP',
                                    'cross_over_date': pd.Timestamp('2020-04-27').tz_localize('UTC'), 
                                    'batch':np.nan, 
                                    'first_plant_based_mention': 'Vegan Cupcake',
                                    'promo_name':'Vegan Cupcake'}]).set_index('location_id') 
d6 = pd.DataFrame([{'location_id': 'C0BE4NDSW26QN',
                    'cross_over_date': pd.Timestamp('2018-05-22').tz_localize('UTC'), 
                                    'batch':np.nan, 
                                    'first_plant_based_mention': 'Impossible Burg',
                                    'promo_name':'Impossible Burg'}]).set_index('location_id') 
        
        
                                                                            
# ED5 bacon potential other dates pd.Timestamp('2021-11-01') 
# ED5 potential vegan sandwich intros pd.Timestamp('2020-10-10') pd.Timestamp('2020-11-07') pd.Timestamp('2020-10-17')

pd.set_option('display.max_colwidth', None) # default is 50

non_food = ["Tres Hombres", 
            "Blue Spirulina", 
            "Catering Utensils Per Person",
            "Holographic Sticker",
            "Bookmark",
            "Print - Large",
            "Collagen Packet",
            "16 New Homestead",
            "13 Glen Hazel *",
            "Eebc Neighborhood Tasting Case",
            "Lucky Charm City", 
            "Lavender Dreams",
            "2$ Special",
            "Oktoberfestspecial",
            "Bingo",
            "The East Enders' Survival Kit"]

sales_and_menu_data_food  = {}
for loc_id, df in sales_and_menu_data.items():
    df = (df
          .query('item_type != "Drink"')
          .query('~dish_category.isin(["Alcohol","Soda","Coffee & Tea", "Water", "Juice","Sports & Health Drink", "Dairy Drink"])')
          .query('~item_name.isin(@non_food)')
          )
    if loc_id == 'SRQS8F7JWA9MZ':
        df = pd.read_parquet('data/3_data_parquet_relabeled/2_consolidated/SRQS8F7JWA9MZ_sales_and_menu.parquet')
    
    if loc_id == '2HRX9P6HKXA8V':
        df = pd.read_parquet('data/3_data_parquet_relabeled/2_consolidated/2HRX9P6HKXA8V_sales_and_menu.parquet')
        
    if loc_id == 'JHDN7CF1C03X5':
        df = pd.read_parquet('data/3_data_parquet_relabeled/2_consolidated/JHDN7CF1C03X5_sales_and_menu.parquet')
        
    if loc_id == 'ED5J990H5VAZT':
        df = pd.read_parquet('data/3_data_parquet_relabeled/2_consolidated/ED5J990H5VAZT_sales_and_menu.parquet')
        
    if loc_id == 'W8T41JZK0ZMEP':
        df = pd.read_parquet('data/3_data_parquet_relabeled/2_consolidated/W8T41JZK0ZMEP_sales_and_menu.parquet')
        
    if loc_id == 'C0BE4NDSW26QN':
        df = pd.read_parquet('data/3_data_parquet_relabeled/2_consolidated/C0BE4NDSW26QN_sales_and_menu.parquet')
    
    sales_and_menu_data_food[loc_id] = df.copy()


days_for_visual = 28
list_of_days = [7, 14, 28]
data_to_plot = None
for days in list_of_days:
    output, potential_confounders_data = get_potential_confounders(sales_and_menu_data_food, before_after_details_true.loc[restaurants_by_4m_coverage], days=days, k=10)
    if days == days_for_visual:
        data_to_plot = potential_confounders_data
    display(output)

for loc_id in restaurants_by_4m_coverage:
    data = data_to_plot[loc_id]
    #plot_dish_time_series(data, loc_id, before_after_details_true)

# print(d1.loc['ED5J990H5VAZT','first_plant_based_mention'])
# for days in list_of_days:
#     output, potential_confounders_data = get_potential_confounders(sales_and_menu_data_food, d1, organization = ['ED5J990H5VAZT'], days=days, k=10)
#     if days == days_for_visual:
#         data_to_plot = potential_confounders_data
#     display(output)
# for loc_id in ['ED5J990H5VAZT']:
#     data = data_to_plot[loc_id]
#     plot_dish_time_series(data, loc_id, d1)
    
# print(d2.loc['ED5J990H5VAZT','first_plant_based_mention'])
# for days in list_of_days:
#     output, potential_confounders_data = get_potential_confounders(sales_and_menu_data_food, d2, organization = ['ED5J990H5VAZT'], days=days, k=10)
#     if days == days_for_visual:
#         data_to_plot = potential_confounders_data
#     display(output)
# for loc_id in ['ED5J990H5VAZT']:
#     data = data_to_plot[loc_id]
#     plot_dish_time_series(data, loc_id, d2)
    
# print(d3.loc['ED5J990H5VAZT','first_plant_based_mention'])
# for days in list_of_days:
#     output, potential_confounders_data = get_potential_confounders(sales_and_menu_data_food, d3, organization = ['ED5J990H5VAZT'], days=days, k=10)
#     if days == days_for_visual:
#         data_to_plot = potential_confounders_data
#     display(output)
# for loc_id in ['ED5J990H5VAZT']:
#     data = data_to_plot[loc_id]
#     plot_dish_time_series(data, loc_id, d3)
    
# print(d4.loc['JHDN7CF1C03X5','first_plant_based_mention'])
# for days in list_of_days:
#     output, potential_confounders_data = get_potential_confounders(sales_and_menu_data_food, d4, organization = ['JHDN7CF1C03X5'], days=days, k=10)
#     if days == days_for_visual:
#         data_to_plot = potential_confounders_data
#     display(output)
# for loc_id in ['JHDN7CF1C03X5']:
#     data = data_to_plot[loc_id]
#     plot_dish_time_series(data, loc_id, d4)
    
print(d5.loc['W8T41JZK0ZMEP','first_plant_based_mention'])
for days in list_of_days:
    output, potential_confounders_data = get_potential_confounders(sales_and_menu_data_food, d5, organization = ['W8T41JZK0ZMEP'], days=days, k=10)
    if days == days_for_visual:
        data_to_plot = potential_confounders_data
    display(output)
for loc_id in ['W8T41JZK0ZMEP']:
    data = data_to_plot[loc_id]
    plot_dish_time_series(data, loc_id, d5)
    
print(d6.loc['C0BE4NDSW26QN','first_plant_based_mention'])
for days in list_of_days:
    output, potential_confounders_data = get_potential_confounders(sales_and_menu_data_food, d6, organization = ['C0BE4NDSW26QN'], days=days, k=10)
    if days == days_for_visual:
        data_to_plot = potential_confounders_data
    display(output)
for loc_id in ['C0BE4NDSW26QN']:
    data = data_to_plot[loc_id]
    plot_dish_time_series(data, loc_id, d6)
    
    


In [ ]:
# have_beyond_sausage = sales_and_menu_data['V3Q26BHF3SE2H'].query('item_name == "Beyond Sausage" or item_modifications.str.contains("Beyond Sausage")')['item_name'].value_counts().index.tolist()
# sales_and_menu_data['V3Q26BHF3SE2H'].query('item_name == "Beyond Sausage" or item_modifications.str.contains("Beyond Sausage")')['item_name'].value_counts()
# sales_and_menu_data['V3Q26BHF3SE2H'].query('item_name.isin(@have_beyond_sausage)')['item_name'].value_counts()
# plot_dish_time_series(sales_and_menu_data_food['V3Q26BHF3SE2H'], 'V3Q26BHF3SE2H', before_after_details_true)